# 📊 Dashboard ElectroMarket en Python · versión completa

> Esta es la **versión extendida** después del taller: 15 gráficas, curva de tendencia, mapas de calor de tardanzas, subcategorías y correlación, histograma con campana de Gauss y página web de una sola pantalla. El notebook original del taller es `Dashboard_ElectroMarket.ipynb`.

Este notebook convierte la práctica de Excel de **ElectroMarket** en un dashboard hecho con código.
Hace lo mismo que tus tablas dinámicas y agrega los KPIs del negocio:

- **Panel de KPIs** con indicadores (▲▼ contra el año anterior) y velocímetros con meta.
- **10 gráficas de 10 tipos distintos:** barras, líneas, anillo, columnas 100%, burbujas, mapa, treemap, mapa de calor, cascada y cajas.
- **Dashboard con filtros** de Mercado, Año y Segmento.

**Cómo usarlo:** corre las celdas en orden (▶ o `Shift + Enter`).
Pon tu Excel en la misma carpeta que este notebook con el nombre `ElectroMarket.xlsx` y elige el entorno del taller en **Select Kernel** (ver README).

## ☁️ ¿Lo abriste en Google Colab?
Corre esta celda primero. En Colab descarga el proyecto de GitHub (con el Excel) y activa los filtros.
En tu computadora no hace nada.

In [ ]:
import sys, os, subprocess

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    USUARIO, REPO = "KrystalRou", "Taller_Bob_Dashboard_26"
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.isdir(REPO):
            subprocess.run(["git", "clone", "-q", f"https://github.com/{USUARIO}/{REPO}.git"], check=True)
        os.chdir(REPO)
    from google.colab import output
    output.enable_custom_widget_manager()   # para que funcionen los filtros
    if not os.path.exists("ElectroMarket.xlsx"):
        from google.colab import files
        print("📤 Sube el Excel de la práctica (se guardará como ElectroMarket.xlsx)")
        subido = files.upload()
        os.rename(next(iter(subido)), "ElectroMarket.xlsx")
    print("☁️ Listo para Colab ✅")
else:
    print("💻 Corriendo en tu computadora: no hace falta preparar nada ✅")

## 0. Librerías
`pandas` maneja la tabla y `plotly` hace las gráficas interactivas. `ipywidgets` agrega los filtros (menús desplegables).

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

pd.set_option("display.float_format", "{:,.2f}".format)
print("Librerías listas ✅")

## 1. Cargar el Excel
El Excel debe estar en la misma carpeta que este notebook. Si tu archivo se llama distinto, cambia el nombre en `ARCHIVO`.

In [ ]:
from pathlib import Path

ARCHIVO = "ElectroMarket.xlsx"      # cambia el nombre si tu archivo se llama distinto

if not Path(ARCHIVO).exists():
    raise FileNotFoundError(f"No encuentro {ARCHIVO}. Ponlo en la misma carpeta que este notebook: {Path.cwd()}")
print("Archivo encontrado ✅:", ARCHIVO)

La hoja con los datos se llama **Ventas Supermercado**. Si no existe, el código usa la primera hoja.

In [ ]:
hojas = pd.ExcelFile(ARCHIVO).sheet_names
print("Hojas del archivo:", hojas)

HOJA = "Ventas Supermercado" if "Ventas Supermercado" in hojas else hojas[0]
crudo = pd.read_excel(ARCHIVO, sheet_name=HOJA)
print("Leí la hoja:", HOJA, "→", crudo.shape[0], "filas y", crudo.shape[1], "columnas")
crudo.head()

## 2. Encontrar las columnas
Los nombres de las columnas pueden venir con o sin acento (`Método de Envio`, `Metodo Envío`...).
Esta función los compara **sin acentos ni mayúsculas**, así el código funciona aunque cambie un poquito el nombre.

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.lower().replace("_", " ").split())

def buscar_columna(df, *opciones, obligatoria=True):
    mapa = {normalizar(c): c for c in df.columns}
    for op in opciones:                      # 1) nombre exacto
        if normalizar(op) in mapa:
            return mapa[normalizar(op)]
    for op in opciones:                      # 2) que lo contenga
        for n, original in mapa.items():
            if normalizar(op) in n:
                return original
    if obligatoria:
        raise KeyError(f"No encontré la columna {opciones}. Columnas: {list(df.columns)}")
    return None

COL = {
    "venta":     buscar_columna(crudo, "ID Venta", obligatoria=False)
                 or buscar_columna(crudo, "Número Venta", "Numero Venta", obligatoria=False),
    "mercado":   buscar_columna(crudo, "Mercado"),
    "pais":      buscar_columna(crudo, "País", "Pais"),
    "ventas":    buscar_columna(crudo, "Ventas"),
    "utilidad":  buscar_columna(crudo, "Utilidad"),
    "prioridad": buscar_columna(crudo, "Prioridad Envio", "Prioridad"),
    "envio":     buscar_columna(crudo, "Método de Envio2", "Metodo de Envio2", obligatoria=False)
                 or buscar_columna(crudo, "Método de Envio", "Metodo Envio"),
    "segmento":  buscar_columna(crudo, "Segmento", obligatoria=False),
    "fecha":     buscar_columna(crudo, "Fecha Salida", obligatoria=False),
    # Columnas extra para los KPIs de operación (si no existen, esas gráficas se saltan)
    "categoria": buscar_columna(crudo, "Categoría", "Categoria", obligatoria=False),
    "subcat":    buscar_columna(crudo, "Sub-Categoría", "Subcategoria", "Sub-Categoria", obligatoria=False),
    "descuento": buscar_columna(crudo, "Descuento", obligatoria=False),
    "status":    buscar_columna(crudo, "Status Llegada", obligatoria=False),
    "dias":      buscar_columna(crudo, "Días Transcurridos", "Dias Transcurridos", obligatoria=False),
    "costo_env": buscar_columna(crudo, "Costo Envío", "Costo Envio", obligatoria=False),
}
for k, v in COL.items():
    print(f"{k:10s} → {v}")

## 3. Limpiar los datos
- Quita filas vacías.
- Arregla el **método de envío** (espacios de más y mayúsculas distintas, por ejemplo `standard class ` → `Standard Class`).
- Arma la **fecha**: usa `Fecha Salida`, y si no existe, la construye con Día, Mes y Año de salida (como la función `FECHA` de Excel).

In [ ]:
df = pd.DataFrame({
    "Mercado":   crudo[COL["mercado"]],
    "País":      crudo[COL["pais"]],
    "Ventas":    pd.to_numeric(crudo[COL["ventas"]], errors="coerce"),
    "Utilidad":  pd.to_numeric(crudo[COL["utilidad"]], errors="coerce"),
    "Prioridad": crudo[COL["prioridad"]],
    "Envío":     crudo[COL["envio"]],
    "Segmento":  crudo[COL["segmento"]] if COL["segmento"] else "Todos",
})
if COL["venta"]:
    df["Pedido"] = crudo[COL["venta"]]
if COL["categoria"]: df["Categoría"]   = crudo[COL["categoria"]].astype(str).str.strip()
if COL["subcat"]:    df["Sub-Categoría"] = crudo[COL["subcat"]].astype(str).str.strip()
if COL["descuento"]: df["Descuento"]   = pd.to_numeric(crudo[COL["descuento"]], errors="coerce")
if COL["status"]:    df["Llegada"]     = crudo[COL["status"]].astype(str).str.strip()
if COL["dias"]:      df["Días"]        = pd.to_numeric(crudo[COL["dias"]], errors="coerce")
if COL["costo_env"]: df["Costo Envío"] = pd.to_numeric(crudo[COL["costo_env"]], errors="coerce")

# Fecha
if COL["fecha"]:
    df["Fecha"] = pd.to_datetime(crudo[COL["fecha"]], errors="coerce", dayfirst=True)
else:
    d = buscar_columna(crudo, "Dia Salida", "Día Salida")
    m = buscar_columna(crudo, "Mes Salida")
    a = buscar_columna(crudo, "Año Salida", "Ano Salida")
    df["Fecha"] = pd.to_datetime(dict(year=crudo[a], month=crudo[m], day=crudo[d]), errors="coerce")

# Limpieza de texto
for c in ["Mercado", "País", "Prioridad", "Envío", "Segmento"]:
    df[c] = df[c].astype(str).str.strip()
df["Envío"] = df["Envío"].str.title()

antes = len(df)
df = df.dropna(subset=["Ventas", "Fecha"])
df["Año"] = df["Fecha"].dt.year
print(f"Filas útiles: {len(df):,}  (quité {antes - len(df):,} vacías o con errores)")
print("Métodos de envío:", sorted(df["Envío"].unique()))
df.head()

## 4. Panel de KPIs: indicadores y velocímetros 🎯
Los KPIs comparan **el último año contra el año anterior**:
- **Indicadores** con flecha verde ▲ si mejoró o roja ▼ si empeoró: ventas, utilidad, pedidos, ticket promedio, días de entrega y costo de envío.
- **Velocímetros** (*gauges*) con una **meta**: margen de utilidad y entregas a tiempo. La raya negra es la meta.

In [ ]:
META_MARGEN = 12      # % de margen que queremos alcanzar
META_A_TIEMPO = 95    # % de entregas a tiempo que queremos alcanzar

def calcular(d):
    ventas, utilidad = d["Ventas"].sum(), d["Utilidad"].sum()
    pedidos = d["Pedido"].nunique() if "Pedido" in d else len(d)
    return {
        "ventas": ventas, "utilidad": utilidad, "pedidos": pedidos,
        "ticket": ventas / pedidos if pedidos else 0,
        "margen": utilidad / ventas * 100 if ventas else 0,
        "a_tiempo": (d["Llegada"].str.lower() == "on time").mean() * 100 if "Llegada" in d else None,
        "dias": d["Días"].mean() if "Días" in d else None,
        "costo_env": d["Costo Envío"].sum() if "Costo Envío" in d else None,
    }

def panel_kpis(actual, previo, titulo):
    a = calcular(actual)
    p = calcular(previo) if len(previo) else None
    fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2,
                        vertical_spacing=0.25, horizontal_spacing=0.06)

    def numero(clave, nombre, fila, col, prefijo="", formato=",.0f", menos_es_mejor=False):
        if a[clave] is None:
            return
        delta = None
        if p and p[clave]:
            delta = dict(reference=p[clave], relative=True, valueformat=".1%",
                         increasing_color="#d62728" if menos_es_mejor else "#2ca02c",
                         decreasing_color="#2ca02c" if menos_es_mejor else "#d62728")
        fig.add_trace(go.Indicator(mode="number+delta" if delta else "number", value=a[clave],
                                   number=dict(prefix=prefijo, valueformat=formato, font_size=34),
                                   delta=delta, title=dict(text=nombre, font_size=15)), fila, col)

    def velocimetro(clave, nombre, fila, col, meta, maximo):
        if a[clave] is None:
            return
        fig.add_trace(go.Indicator(
            mode="gauge+number+delta", value=a[clave],
            number=dict(suffix="%", valueformat=".1f", font_size=30),
            delta=dict(reference=meta, valueformat=".1f", suffix=" pts vs meta"),
            title=dict(text=nombre, font_size=15),
            gauge=dict(axis=dict(range=[0, maximo]), bar=dict(color="#1f3b73"),
                       steps=[dict(range=[0, meta * 0.8], color="#f8d7da"),
                              dict(range=[meta * 0.8, meta], color="#fff3cd"),
                              dict(range=[meta, maximo], color="#d4edda")],
                       threshold=dict(line=dict(color="black", width=4), value=meta))), fila, col)

    numero("ventas", "Ventas", 1, 1, "$", ".3s")
    numero("utilidad", "Utilidad", 1, 2, "$", ".3s")
    numero("pedidos", "Pedidos", 1, 3)
    numero("ticket", "Ticket promedio", 1, 4, "$")
    velocimetro("margen", "Margen de utilidad", 2, 1, META_MARGEN, 30)
    velocimetro("a_tiempo", "Entregas a tiempo", 2, 2, META_A_TIEMPO, 100)
    numero("dias", "Días de entrega", 2, 3, formato=".1f", menos_es_mejor=True)
    numero("costo_env", "Costo de envío", 2, 4, "$", ".3s", menos_es_mejor=True)
    return fig.update_layout(title=titulo, height=480, margin=dict(t=90, b=20))

def periodo(base, anio=None):
    """Regresa (datos del año, datos del año anterior, título)."""
    anio = anio or int(base["Año"].max())
    actual, previo = base[base["Año"] == anio], base[base["Año"] == anio - 1]
    titulo = f"KPIs de {anio}" + (f" comparados con {anio - 1}" if len(previo) else "")
    return actual, previo, titulo

panel_kpis(*periodo(df)).show()

## 5. Quince gráficas, muchos tipos distintos 📈
Cada gráfica es una **función** que recibe la tabla (`d`) y regresa la figura, así las reutilizamos en el dashboard con filtros.
Las primeras 5 son tus **tablas dinámicas** de Excel; las otras 5 responden a los **KPIs**:

| # | Pregunta | Tipo de gráfica |
|---|---|---|
| 1 | ¿Qué mercado vende más? | Barras horizontales |
| 2 | ¿Cómo cambian las ventas en el tiempo? | Líneas |
| 3 | ¿Qué método de envío prefieren? | Anillo |
| 4 | ¿Cómo se reparten las prioridades en cada mercado? | Columnas 100% apiladas |
| 5 | ¿Vender más significa ganar más? | Burbujas (dispersión) |
| 6 | ¿En qué países se gana o se pierde dinero? | Mapa |
| 7 | ¿Qué productos venden más y cuáles son rentables? | Treemap (mosaico) |
| 8 | ¿Qué combinación mercado–categoría tiene mejor margen? | Mapa de calor |
| 9 | ¿Cómo creció la utilidad periodo a periodo? | Cascada |
| 10 | ¿Los descuentos causan pérdidas? | Cajas (box plot) |

Recordatorio: `groupby(...)` = campo en **Filas**, `.sum()` = **Suma de** en **Valores**, `pivot_table(columns=...)` = campo en **Columnas**.

In [ ]:
COLORES = px.colors.qualitative.Set2
MESES_CORTOS = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]

# 1) Barras horizontales: ventas por mercado
def g_mercado(d):
    t = d.groupby("Mercado", as_index=False)["Ventas"].sum().sort_values("Ventas")
    fig = px.bar(t, x="Ventas", y="Mercado", orientation="h", text_auto=".3s",
                 title="1. Ventas por mercado · Barras", color_discrete_sequence=[COLORES[0]])
    return fig.update_layout(xaxis_title="Ventas (USD)", yaxis_title="")

# 2) Líneas: ventas mensuales, una línea por año
def g_mensual(d):
    t = d.groupby([d["Fecha"].dt.year.rename("Año"), d["Fecha"].dt.month.rename("Mes")])["Ventas"].sum().reset_index()
    t["Año"] = t["Año"].astype(str)
    fig = px.line(t, x="Mes", y="Ventas", color="Año", markers=True,
                  title="2. Ventas mensuales por año · Líneas", color_discrete_sequence=COLORES)
    return fig.update_xaxes(tickvals=list(range(1, 13)), ticktext=MESES_CORTOS).update_layout(yaxis_title="Ventas (USD)")

# 3) Anillo: método de envío
def g_envio(d):
    t = d.groupby("Envío", as_index=False)["Ventas"].sum()
    fig = px.pie(t, names="Envío", values="Ventas", hole=0.5,
                 title="3. Ventas por método de envío · Anillo", color_discrete_sequence=COLORES)
    return fig.update_traces(textinfo="percent+label")

# 4) Columnas 100% apiladas: prioridad por mercado
ORDEN_PRIORIDAD = ["Critical", "High", "Medium", "Low"]
def g_prioridad(d):
    t = d.pivot_table(index="Mercado", columns="Prioridad", values="Ventas", aggfunc="sum", fill_value=0)
    t = (t.div(t.sum(axis=1), axis=0) * 100).reset_index()
    largo = t.melt(id_vars="Mercado", var_name="Prioridad", value_name="%")
    orden = [p for p in ORDEN_PRIORIDAD if p in largo["Prioridad"].unique()] or None
    fig = px.bar(largo, x="Mercado", y="%", color="Prioridad", text_auto=".0f",
                 category_orders={"Prioridad": orden} if orden else None,
                 title="4. Prioridad de envío por mercado · Columnas 100%", color_discrete_sequence=COLORES)
    return fig.update_layout(barmode="stack", yaxis_title="% de las ventas", yaxis_range=[0, 100])

# 5) Burbujas: ventas vs. utilidad por país (tamaño = número de pedidos)
def g_dispersion(d):
    t = d.groupby(["País", "Mercado"]).agg(Ventas=("Ventas", "sum"), Utilidad=("Utilidad", "sum"),
                                            Pedidos=("Pedido" if "Pedido" in d else "Ventas", "count")).reset_index()
    fig = px.scatter(t, x="Ventas", y="Utilidad", color="Mercado", size="Pedidos", size_max=40,
                     hover_name="País", title="5. Ventas vs. utilidad por país · Burbujas",
                     color_discrete_sequence=COLORES)
    if len(t) > 1:  # línea de tendencia (curva: polinomio de grado 3) con numpy
        coef = np.polyfit(t["Ventas"], t["Utilidad"], 3)
        xs = np.linspace(t["Ventas"].min(), t["Ventas"].max(), 200)
        fig.add_scatter(x=xs, y=np.polyval(coef, xs), mode="lines", name="Tendencia", line=dict(color="red", dash="dot"))
        r = np.corrcoef(t["Ventas"], t["Utilidad"])[0, 1]
        fig.add_annotation(text=f"r = {r:.2f}", xref="paper", yref="paper", x=0.02, y=0.98, showarrow=False)
    fig.add_hline(y=0, line_color="red", line_width=1)   # abajo de la línea roja = pérdidas
    return fig.update_layout(xaxis_title="Ventas (USD)", yaxis_title="Utilidad (USD)")

# 6) Mapa: utilidad por país (rojo = pérdida, verde = ganancia)
def g_mapa(d):
    t = d.groupby("País", as_index=False)[["Ventas", "Utilidad"]].sum()
    lim = t["Utilidad"].abs().quantile(0.95)
    fig = px.choropleth(t, locations="País", locationmode="country names", color="Utilidad",
                        hover_data={"Ventas": ":$,.0f", "Utilidad": ":$,.0f"},
                        color_continuous_scale="RdYlGn", range_color=[-lim, lim],
                        title="6. Utilidad por país · Mapa")
    return fig.update_geos(showframe=False, projection_type="natural earth").update_layout(margin=dict(l=0, r=0))

# 7) Treemap: categoría → sub-categoría (tamaño = ventas, color = margen)
def g_treemap(d):
    if "Categoría" not in d:
        return None
    niveles = ["Categoría", "Sub-Categoría"] if "Sub-Categoría" in d else ["Categoría"]
    t = d.groupby(niveles, as_index=False)[["Ventas", "Utilidad"]].sum()
    t["Margen %"] = t["Utilidad"] / t["Ventas"] * 100
    fig = px.treemap(t, path=[px.Constant("Todos")] + niveles, values="Ventas", color="Margen %",
                     color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
                     title="7. Ventas y margen por categoría · Treemap")
    return fig.update_traces(texttemplate="<b>%{label}</b><br>$%{value:,.3s}").update_layout(margin=dict(l=10, r=10))

# 8) Mapa de calor: margen % por mercado y categoría
def g_calor(d):
    if "Categoría" not in d:
        return None
    v = d.pivot_table(index="Mercado", columns="Categoría", values="Ventas", aggfunc="sum")
    u = d.pivot_table(index="Mercado", columns="Categoría", values="Utilidad", aggfunc="sum")
    fig = px.imshow((u / v * 100).round(1), text_auto=".1f", aspect="auto",
                    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
                    title="8. Margen % por mercado y categoría · Mapa de calor")
    return fig.update_layout(coloraxis_colorbar_title="Margen %", xaxis_title="", yaxis_title="")

# 9) Cascada: cómo cambió la utilidad periodo a periodo
def g_cascada(d):
    if d["Año"].nunique() > 1:
        t = d.groupby("Año")["Utilidad"].sum()
        etiquetas = [str(a) for a in t.index]
    else:   # si filtras un solo año, la cascada va por trimestre
        t = d.groupby(d["Fecha"].dt.quarter)["Utilidad"].sum()
        etiquetas = [f"T{q}" for q in t.index]
    valores = [t.iloc[0]] + list(t.diff().iloc[1:]) + [t.iloc[-1]]
    medidas = ["absolute"] + ["relative"] * (len(t) - 1) + ["total"]
    fig = go.Figure(go.Waterfall(x=etiquetas + [f"Cierre {etiquetas[-1]}"], y=valores, measure=medidas,
                                 text=[f"${v:,.0f}" for v in valores], textposition="outside",
                                 increasing_marker_color="#2ca02c", decreasing_marker_color="#d62728",
                                 totals_marker_color="#1f3b73"))
    return fig.update_layout(title="9. Cómo cambió la utilidad · Cascada", yaxis_title="Utilidad (USD)", showlegend=False)

# 10) Cajas: utilidad por nivel de descuento
def g_descuento(d):
    if "Descuento" not in d:
        return None
    rangos = pd.cut(d["Descuento"], bins=[-0.01, 0, 0.2, 0.4, 1],
                    labels=["Sin descuento", "1–20%", "21–40%", "Más de 40%"])
    t = d.assign(Rango=rangos)
    fig = px.box(t, x="Rango", y="Utilidad", color="Rango", points=False,
                 category_orders={"Rango": list(rangos.cat.categories)},
                 title="10. Utilidad por venta según el descuento · Cajas", color_discrete_sequence=COLORES)
    bajo, alto = t["Utilidad"].quantile([0.02, 0.98])
    fig.add_hline(y=0, line_color="red", line_width=1)
    return fig.update_layout(yaxis_range=[bajo, alto], showlegend=False, xaxis_title="Descuento",
                             yaxis_title="Utilidad por venta (USD)")

# 11) Mapa de calor: % de envíos que llegan tarde por mercado y prioridad
def g_tardanzas(d):
    if "Llegada" not in d or "Mercado" not in d or "Prioridad" not in d:
        return None
    orden = [p for p in ["Critical", "High", "Medium", "Low"] if p in d["Prioridad"].unique()]
    tarde = d.assign(_late=d["Llegada"].str.lower() == "late")
    t = tarde.pivot_table(index="Mercado", columns="Prioridad", values="_late", aggfunc="mean")[orden] * 100
    fig = px.imshow(t.round(1), text_auto=".0f", aspect="auto",
                    color_continuous_scale="Reds",
                    title="11. % de envíos que llegan tarde · Mapa de calor")
    return fig.update_layout(coloraxis_colorbar_title="% tarde", xaxis_title="", yaxis_title="")

# 12) Áreas apiladas al 100%: cómo cambia el peso de cada mercado con el tiempo
#     (como las gráficas de física donde cada especie ocupa una fracción según la energía)
def g_areas(d):
    trimestre = d["Fecha"].dt.to_period("Q").astype(str).rename("Trimestre")
    t = d.groupby([trimestre, "Mercado"])["Ventas"].sum().reset_index()
    t["Porcentaje"] = t["Ventas"] / t.groupby("Trimestre")["Ventas"].transform("sum") * 100
    orden = d.groupby("Mercado")["Ventas"].sum().sort_values(ascending=False).index.tolist()
    fig = px.area(t, x="Trimestre", y="Porcentaje", color="Mercado", line_shape="spline",
                  category_orders={"Mercado": orden}, color_discrete_sequence=COLORES,
                  title="12. Peso de cada mercado en el tiempo · Áreas apiladas 100%")
    return fig.update_layout(yaxis_title="% de las ventas", yaxis_range=[0, 100], xaxis_title="")

# 13) Mapa de calor: utilidad de cada subcategoría en cada mercado (verde gana, rojo pierde)
def g_subcat_mercado(d):
    if "Sub-Categoría" not in d:
        return None
    t = d.pivot_table(index="Sub-Categoría", columns="Mercado", values="Utilidad", aggfunc="sum").fillna(0)
    t = t.loc[t.sum(axis=1).sort_values(ascending=False).index]   # las que más ganan arriba
    lim = t.abs().max().max()
    fig = px.imshow(t.round(0), text_auto=".2s", aspect="auto", color_continuous_scale="RdYlGn",
                    zmin=-lim, zmax=lim, title="13. Utilidad por subcategoría y mercado · Mapa de calor")
    return fig.update_layout(coloraxis_colorbar_title="Utilidad", xaxis_title="", yaxis_title="")

# 14) Correlación: ¿qué se mueve junto con la utilidad en cada tipo de producto?
#     +1 = suben juntas (azul) · 0 = no tienen relación · -1 = una sube y la otra baja (rojo)
def g_correlacion(d):
    if "Sub-Categoría" not in d:
        return None
    cols = [c for c in ["Ventas", "Descuento", "Costo Envío", "Días"] if c in d]
    filas = {sub: {c: g["Utilidad"].corr(g[c]) for c in cols} for sub, g in d.groupby("Sub-Categoría")}
    t = pd.DataFrame(filas).T
    if "Descuento" in t:
        t = t.sort_values("Descuento")          # las más dañadas por el descuento arriba
    t = t.rename(columns={"Días": "Días de entrega"})
    fig = px.imshow(t.round(2), text_auto=".2f", aspect="auto", color_continuous_scale="RdBu",
                    zmin=-1, zmax=1, title="14. Correlación de la utilidad por tipo de producto · Mapa de calor")
    return fig.update_layout(coloraxis_colorbar_title="r", yaxis_title="",
                             xaxis_title="Correlación de la utilidad con…")

# 15) Histograma: el valor de cada venta sigue una campana de Gauss... en escala logarítmica
#     (muchas ventas chicas y pocas enormes = distribución log-normal, muy común en negocios)
def g_histograma(d):
    x = np.log10(d.loc[d["Ventas"] > 0, "Ventas"])
    if len(x) < 2:
        return None
    mu, sigma = x.mean(), x.std()
    fig = px.histogram(x=x, nbins=60, histnorm="probability density", color_discrete_sequence=[COLORES[2]],
                       title="15. ¿Cuánto vale cada venta? · Histograma con curva de Gauss")
    xs = np.linspace(x.min(), x.max(), 200)
    gauss = np.exp(-((xs - mu) / sigma) ** 2 / 2) / (sigma * np.sqrt(2 * np.pi))
    fig.add_scatter(x=xs, y=gauss, mode="lines", name="Curva de Gauss", line=dict(color="red", width=3))
    ticks = [t for t in [1, 10, 100, 1000, 10000] if x.min() - 0.5 <= np.log10(t) <= x.max() + 0.5]
    fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"${t:,}" for t in ticks])
    fig.add_annotation(text=f"Venta típica: ${10 ** mu:,.0f}", xref="paper", yref="paper",
                       x=0.02, y=0.98, showarrow=False)
    return fig.update_layout(xaxis_title="Valor de la venta (USD, escala logarítmica)",
                             yaxis_title="Densidad", showlegend=False, bargap=0.05)

GRAFICAS = [g_mercado, g_mensual, g_envio, g_prioridad, g_dispersion,
            g_mapa, g_treemap, g_calor, g_cascada, g_descuento, g_tardanzas, g_areas,
            g_subcat_mercado, g_correlacion, g_histograma]

def mostrar(g, d, alto=440):
    fig = g(d)
    if fig is not None:
        fig.update_layout(height=alto, template="plotly_white").show()

for g in GRAFICAS:
    mostrar(g, df)

## 6. Dashboard interactivo con filtros 🎛️
Elige **Mercado**, **Año** y **Segmento**: el panel de KPIs y las 10 gráficas se actualizan solos
(como las *segmentaciones* conectadas a todas las tablas dinámicas en Excel).
Si eliges un año, los KPIs lo comparan con el año anterior.

In [ ]:
def opciones(col):
    return ["Todos"] + sorted(df[col].dropna().astype(str).unique())

f_mercado  = widgets.Dropdown(options=opciones("Mercado"), description="Mercado")
f_anio     = widgets.Dropdown(options=["Todos"] + sorted(df["Año"].unique().tolist()), description="Año")
f_segmento = widgets.Dropdown(options=opciones("Segmento"), description="Segmento")
salida = widgets.Output()

def sin_anio():
    d = df
    if f_mercado.value  != "Todos": d = d[d["Mercado"] == f_mercado.value]
    if f_segmento.value != "Todos": d = d[d["Segmento"] == f_segmento.value]
    return d

def filtrar():
    d = sin_anio()
    return d if f_anio.value == "Todos" else d[d["Año"] == f_anio.value]

def dibujar(_=None):
    d = filtrar()
    with salida:
        salida.clear_output(wait=True)
        if d.empty:
            print("No hay datos con esos filtros 🙃")
            return
        anio = None if f_anio.value == "Todos" else f_anio.value
        panel_kpis(*periodo(sin_anio(), anio)).show()
        for g in GRAFICAS:
            mostrar(g, d, alto=420)

for w in (f_mercado, f_anio, f_segmento):
    w.observe(dibujar, names="value")

display(HTML("<h2 style='font-family:sans-serif;color:#1f3b73'>📊 Dashboard ElectroMarket</h2>"),
        widgets.HBox([f_mercado, f_anio, f_segmento]), salida)
dibujar()

## 7. Hallazgos automáticos
Python también puede **escribir las conclusiones** buscando los datos clave.

In [ ]:
por_mercado = df.groupby("Mercado")["Ventas"].sum().sort_values(ascending=False)
por_pais = df.groupby("País")[["Ventas", "Utilidad"]].sum()
envio = df.groupby("Envío")["Ventas"].sum()
mes_top = df.groupby(df["Fecha"].dt.month)["Ventas"].sum().idxmax()
perdidas = por_pais[por_pais["Utilidad"] < 0].sort_values("Utilidad")
anual = df.groupby("Año")["Ventas"].sum()

print(f"🏆 Mercado que más vende: {por_mercado.index[0]} (${por_mercado.iloc[0]:,.0f})")
print(f"🌎 País que más vende: {por_pais['Ventas'].idxmax()}")
print(f"🚚 Envío más usado: {envio.idxmax()} ({envio.max() / envio.sum():.0%} de las ventas)")
MESES = ["enero","febrero","marzo","abril","mayo","junio","julio","agosto","septiembre","octubre","noviembre","diciembre"]
print(f"📅 Mes más fuerte (todos los años): {MESES[mes_top - 1]}")
print(f"📈 Ventas por año: " + ", ".join(f"{a}: ${v:,.0f}" for a, v in anual.items()))
print(f"⚠️ Países con pérdidas: {len(perdidas)} de {len(por_pais)}")
if len(perdidas):
    print("   Los 5 con más pérdida:", ", ".join(perdidas.head(5).index))
margen = (df.groupby("Mercado")["Utilidad"].sum() / df.groupby("Mercado")["Ventas"].sum() * 100).sort_values()
print(f"💰 Mercado más rentable: {margen.index[-1]} ({margen.iloc[-1]:.1f}% de margen); menos rentable: {margen.index[0]} ({margen.iloc[0]:.1f}%)")
if "Categoría" in df:
    cat = df.groupby("Categoría")["Utilidad"].sum()
    print(f"📦 Categoría con más utilidad: {cat.idxmax()} (${cat.max():,.0f})")
if "Llegada" in df:
    tarde = df.assign(t=df["Llegada"].str.lower() == "late").groupby("Envío")["t"].mean()
    print(f"⏰ Envío que más llega tarde: {tarde.idxmax()} ({tarde.max():.0%} de sus envíos)")

## 8. Guardar el dashboard como página web
Crea `dashboard_electromarket_completo.html` con el panel de KPIs y las 15 gráficas **en una sola pantalla**, con los filtros que elegiste arriba.
Ábrelo con doble clic: **toca cualquier gráfica** para verla en pantalla completa, usa ◀ ▶ (o las flechas del teclado) para pasar a la siguiente y **Esc** para regresar.

In [ ]:
import sys
import plotly.io as pio

PLANTILLA = r"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Dashboard ElectroMarket</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  * { box-sizing: border-box; }
  body { margin: 0; font-family: -apple-system, "Segoe UI", Roboto, sans-serif; background: #f4f6fa; color: #1c2533; }
  header { height: 56px; display: flex; align-items: center; justify-content: space-between; padding: 0 20px; }
  header h1 { font-size: 20px; margin: 0; color: #1f3b73; }
  header span { font-size: 13px; color: #5b6575; }
  .grid { display: grid; gap: 10px; padding: 0 14px 14px;
          grid-template-columns: repeat(4, 1fr);
          grid-auto-rows: max(170px, calc((100vh - 56px - 14px - 30px) / 4)); }
  .card { background: #fff; border: 1px solid #dfe3ea; border-radius: 10px; padding: 8px 10px 4px;
          display: flex; flex-direction: column; cursor: zoom-in; transition: box-shadow .15s, transform .15s; min-width: 0; }
  .card:hover { box-shadow: 0 6px 18px rgba(31,59,115,.18); transform: translateY(-2px); }
  .card h3 { margin: 0 0 4px; font-size: 13px; font-weight: 600; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
  .card h3::after { content: " ⤢"; color: #8a94a6; }
  .card .plot { flex: 1; min-height: 0; pointer-events: none; }
  .card.kpis { grid-column: span 2; grid-row: span 2; }
  #vista { position: fixed; inset: 0; background: #fff; display: none; flex-direction: column; z-index: 10; }
  #vista.abierta { display: flex; }
  #vista .barra { height: 52px; display: flex; align-items: center; gap: 10px; padding: 0 16px; border-bottom: 1px solid #dfe3ea; }
  #vista .barra h2 { flex: 1; font-size: 17px; margin: 0; }
  #vista button { font-size: 14px; padding: 6px 12px; border: 1px solid #c8cfda; border-radius: 8px; background: #fff; cursor: pointer; }
  #vista button:hover { background: #eef2f8; }
  #grande { flex: 1; min-height: 0; }
  @media (max-width: 900px) {
    .grid { grid-template-columns: repeat(2, 1fr); grid-auto-rows: 220px; }
  }
</style></head>
<body>
<header><h1>📊 Dashboard ElectroMarket</h1><span>__FILTROS__ · toca una gráfica para verla en grande</span></header>
<div class="grid" id="grid"></div>
<div id="vista">
  <div class="barra"><button id="ant">◀</button><h2 id="titulo"></h2><button id="sig">▶</button><button id="cerrar">✕ Cerrar (Esc)</button></div>
  <div id="grande"></div>
</div>
<script>
const FIGURAS = [__KPIS__].concat(__FIGS__);
const titulo = f => (f.layout.title && (f.layout.title.text || f.layout.title)) || "";
const grid = document.getElementById("grid");
const filas = Math.ceil((FIGURAS.length - 1 + 4) / 4);   // 4 columnas; los KPIs ocupan 4 lugares
if (window.innerWidth > 900) grid.style.gridAutoRows = `max(150px, calc((100vh - 100px) / ${filas}))`;
const vista = document.getElementById("vista");
const grande = document.getElementById("grande");
let actual = -1;

FIGURAS.forEach((fig, i) => {
  const card = document.createElement("div");
  card.className = "card" + (i === 0 ? " kpis" : "");
  card.innerHTML = `<h3>${titulo(fig)}</h3><div class="plot"></div>`;
  card.onclick = () => abrir(i);
  grid.appendChild(card);
  const div = card.querySelector(".plot");
  const layout = Object.assign({}, fig.layout, {
    title: null, autosize: true, height: div.clientHeight || 160,
    margin: i === 0 ? {l: 10, r: 10, t: 10, b: 10} : {l: 34, r: 8, t: 6, b: 26},
    showlegend: false, font: {size: 9}
  });
  if (i > 0) layout.annotations = [];
  let data = fig.data;
  if (i === 0) {  // KPIs: números más chicos para que quepan en la miniatura
    data = fig.data.map(t => Object.assign({}, t, {
      number: Object.assign({}, t.number, {font: {size: t.gauge ? 20 : 26}}),
      delta: Object.assign({}, t.delta, {font: {size: 11}}),
      title: Object.assign({}, t.title, {font: {size: 12}})
    }));
  }
  Plotly.newPlot(div, data, layout, {staticPlot: true, responsive: true, displayModeBar: false});
});

function abrir(i) {
  actual = (i + FIGURAS.length) % FIGURAS.length;
  const fig = FIGURAS[actual];
  document.getElementById("titulo").textContent = titulo(fig);
  vista.classList.add("abierta");
  Plotly.purge(grande);
  const layout = Object.assign({}, fig.layout, {title: null, autosize: true, height: null, width: null});
  Plotly.newPlot(grande, fig.data, layout, {responsive: true, displaylogo: false});
}
function cerrar() { vista.classList.remove("abierta"); Plotly.purge(grande); actual = -1; }
document.getElementById("cerrar").onclick = cerrar;
document.getElementById("ant").onclick = () => abrir(actual - 1);
document.getElementById("sig").onclick = () => abrir(actual + 1);
document.addEventListener("keydown", e => {
  if (actual < 0) return;
  if (e.key === "Escape") cerrar();
  if (e.key === "ArrowRight") abrir(actual + 1);
  if (e.key === "ArrowLeft") abrir(actual - 1);
});
</script>
</body></html>"""


def exportar_html(nombre="dashboard_electromarket_completo.html"):
    """Página web: todas las gráficas en una sola pantalla; al tocar una se abre en grande."""
    d = filtrar()   # respeta los filtros elegidos arriba
    anio = None if f_anio.value == "Todos" else f_anio.value
    kpis = panel_kpis(*periodo(sin_anio(), anio)).update_layout(template="plotly_white")
    figuras = [f.update_layout(template="plotly_white") for f in (g(d) for g in GRAFICAS) if f is not None]
    filtros = f"Mercado: {f_mercado.value} · Año: {f_anio.value} · Segmento: {f_segmento.value}"
    pagina = (PLANTILLA
              .replace("__FILTROS__", filtros)
              .replace("__KPIS__", pio.to_json(kpis))
              .replace("__FIGS__", "[" + ",".join(pio.to_json(f) for f in figuras) + "]"))
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(pagina)
    if "google.colab" in sys.modules:   # en Colab, la página se descarga a tu computadora
        from google.colab import files
        files.download(nombre)
    return nombre

print("Guardado ✅:", exportar_html(), "→ ábrelo con doble clic en tu navegador")


## 🚀 Retos para seguir practicando
1. Agrega un filtro de **País** (pista: copia cómo se hizo `f_mercado`).
2. Haz una gráfica de **Top 10 productos por utilidad** con `nlargest(10)`.
3. Cambia la gráfica 2 para que muestre **Utilidad** en lugar de Ventas.
4. Calcula el **margen por sub-categoría** (`Utilidad / Ventas`) y grafícalo en barras.
5. Sube este notebook a tu repo de **GitHub** (Archivo → Guardar una copia en GitHub).